# 05 - EDA and Feature Engineering**Objective:** Explore heart disease data patterns and create new features.**Steps:**1. Statistical summary and class distribution2. Visualizations (distributions, correlations, feature comparison by target)3. Feature scaling4. Feature engineering (interaction and ratio features)5. Save engineered data

In [ ]:
import pandas as pdimport numpy as npfrom pathlib import Pathimport matplotlib.pyplot as pltimport seaborn as snssns.set_style("whitegrid")plt.rcParams["figure.figsize"] = (10, 6)print("Libraries imported successfully")

In [ ]:
# TODO: Load the clean data and inspect its structure# Start by reading data/processed/clean_data.csv into a DataFrame.# Then use .info() to see column dtypes and non-null counts,# .head() to preview a few rows, and .describe() for summary statistics.PROCESSED_DIR = Path("../data/processed")# df = pd.read_csv(PROCESSED_DIR / "clean_data.csv")# print(df.info())# print(df.head())# print(df.describe())

### Statistical Summary & DistributionBefore building any models, it is important to understand the range and spread of your data.The target variable — `target` — is binary (a classification problem).Check the class balance to see if the dataset is imbalanced, which might affect model evaluation.Heart disease data has 13 clinical features covering patient demographics,blood test results, and ECG measurements.

In [ ]:
# TODO: Examine the target variable distribution and feature summary# Use df.describe() to see min, max, mean, and quartiles for all features.# Check the class balance with df["target"].value_counts().## print(df.describe())# print(df["target"].value_counts())

#### A little primer on groupby - `groupby` is a powerful pandas method that allows you to split your data into groups based on some criteria, apply a function to each group, and then combine the results. For example, to see how cholesterol levels differ between patients with and without heart disease, you can do:```pythonchol_by_target = df.groupby("target")["chol"].mean()```- `aggregate` is a method that allows you to apply multiple functions to your grouped data. For example, to get both the mean and standard deviation of cholesterol by target, you can do:```pythonstats_by_target = df.groupby("target")["chol"].aggregate(["mean", "std"])```Aggregate functions can be any function that takes a Series and returns a single value, such as `mean`, `std`, `min`, `max`, etc.Aggregate can be deployed on multiple columns at once, and you can specify different functions for each column if needed.

In [ ]:
# === Executed Example: GroupBy and Aggregate ===# Small inline dataset showing how groupby splits data by target# and compares clinical measurements between classes.import pandas as pddata = pd.DataFrame({    "target": [0, 0, 0, 1, 1, 1],    "age": [55, 60, 58, 65, 70, 62],    "chol": [220, 240, 210, 280, 310, 260],})mean_by_target = data.groupby("target")["chol"].mean()print("Average cholesterol by target:\n", mean_by_target)stats_by_target = data.groupby("target")["age"].agg(["mean", "std", "min", "max"])print("\nAge statistics by target:\n", stats_by_target)

In [ ]:
# === Commented Template: GroupBy and Aggregate ===# Uncomment and adapt to your own dataset.# import pandas as pd# data = pd.DataFrame({#     "group_col": [val1, val1, val2, val2],#     "value_col": [10, 20, 30, 40],# })# grouped = data.groupby("group_col")["value_col"].mean()# stats = data.groupby("group_col")["value_col"].agg(["mean", "std", "min", "max"])

### Missing Value Imputation

Our clean data has no missing values, but the corrupted variants from
`data_injection/` do. Common strategies:

- **Drop rows**: `df.dropna()` — fast, loses samples
- **Mean/Median imputation**: `SimpleImputer(strategy='median')` — preserves sample count
- **KNN imputation**: `KNNImputer()` — estimates from neighbors, more robust
- **Forward fill**: `df.ffill()` — for sequential data

In [ ]:
# === Executed Example: Missing Value Imputation ===
# Using SimpleImputer to fill missing values with the mean.

from sklearn.impute import SimpleImputer

data = pd.DataFrame({
    "age": [55.0, np.nan, 58.0, 65.0, np.nan],
    "chol": [220.0, 240.0, np.nan, 280.0, 310.0],
})

print("Before imputation:")
print(data)

imputer = SimpleImputer(strategy='mean')
imputed = imputer.fit_transform(data)
imputed_df = pd.DataFrame(imputed, columns=data.columns)
print("\nAfter imputation (mean strategy):")
print(imputed_df)
print(f"\nImputed age: {imputer.statistics_[0]:.3f}")
print(f"Imputed chol: {imputer.statistics_[1]:.3f}")

In [ ]:
# TODO: Compare feature distributions by target# Which features best separate no disease (0) from heart disease (1)?# Use boxplots to compare key clinical features.## features_to_plot = ["age", "chol", "thalach", "trestbps", "oldpeak"]# fig, axes = plt.subplots(2, 3, figsize=(14, 10))# for ax, feature in zip(axes.ravel(), features_to_plot):#     sns.boxplot(x="target", y=feature, data=df, ax=ax)#     ax.set_title(f"{feature} by target")# plt.tight_layout()# plt.show()

### VisualizationsVisual exploration helps you spot patterns and relationships that summary statistics miss.Focus on:- How each clinical feature is distributed (histograms)- How features correlate with each other and with the target (heatmap)- Which features separate heart disease patients from healthy ones (boxplots)

In [ ]:
# TODO: Plot histograms for key numerical features# Pick features that are likely to differ between classes.# Use df[features].hist() with bins=30 and a large figure size.# features_to_plot = ["age", "chol", "thalach", "trestbps", "oldpeak"]# df[features_to_plot].hist(bins=30, figsize=(15, 10))# plt.tight_layout()# plt.show()

In [ ]:
# TODO: Create a correlation heatmap# Use df.corr() to compute pairwise correlations between numeric columns.# Then visualize with sns.heatmap() — this quickly shows which features# are strongly (positively or negatively) correlated with the target.## TODO: Identify the top features correlated with target# Sort the correlation values for the target column to see which features# have the strongest relationship with heart disease.# plt.figure(figsize=(12, 10))# sns.heatmap(df.corr(), annot=False, cmap="coolwarm", center=0)# plt.title("Feature Correlation Heatmap")# plt.show()## target_corr = df.corr()["target"].sort_values(ascending=False)# print("Top correlations with target:")# print(target_corr)

### Feature ScalingMany machine learning algorithms (SVR, linear models, neural networks) are sensitive tothe scale of input features. StandardScaler transforms each feature to have mean 0 andstandard deviation 1, which puts all features on equal footing.Tree-based models (Random Forest, XGBoost) do not require scaling since they split onthresholds independently of feature magnitude.

In [ ]:
# === Executed Example: Feature Scaling ===# Small inline dataset showing how StandardScaler transforms features# to have mean ~0 and std ~1.from sklearn.preprocessing import StandardScalerimport pandas as pddata = pd.DataFrame({    "age": [55, 60, 58, 65, 70],    "chol": [220, 240, 210, 280, 310],    "target": [0, 0, 0, 1, 1],})scaler = StandardScaler()scaled_features = scaler.fit_transform(data[["age", "chol"]])scaled_df = pd.DataFrame(scaled_features, columns=["age_scaled", "chol_scaled"])scaled_df["target"] = data["target"]print(scaled_df)print(f"Means after scaling: {scaled_df[['age_scaled', 'chol_scaled']].mean().values}")print(f"Stds after scaling: {scaled_df[['age_scaled', 'chol_scaled']].std().values}")

In [ ]:
from sklearn.preprocessing import StandardScaler# TODO: Separate features and target, then scale the features# Define X as all columns except "target" and y as the target column.## Create a StandardScaler, call fit_transform() on X to compute the mean and std# and return the scaled array in one step.## After scaling, verify that each feature has mean ~0 and std ~1.# X = df.drop("target", axis=1)# y = df["target"]# scaler = StandardScaler()# X_scaled = scaler.fit_transform(X)# print(f"Mean after scaling (first 5): {X_scaled.mean(axis=0)[:5]}")# print(f"Std after scaling (first 5): {X_scaled.std(axis=0)[:5]}")# print(f"All means near zero: {np.allclose(X_scaled.mean(axis=0), 0, atol=1e-10)}")

### Feature EngineeringNew features derived from existing columns can capture interactions and non-linear relationships.Good candidates for heart disease:- **Age × Cholesterol**: Combined risk factor — older age + high cholesterol amplifies risk- **Max heart rate ratio**: `thalach / (220 - age)` — achieved vs predicted max heart rate- **Oldpeak × Slope**: ST depression magnitude weighted by slope directionBe careful with division by zero — add a small epsilon or +1 to the denominator.#### NoteIn pandas, you can create interaction features like this:```pythondf["feature1_feature2"] = df["feature1"] * df["feature2"]```

In [ ]:
# === Executed Example: Interaction Features ===# Multiplication and ratio on a small inline dataset.import pandas as pddata = pd.DataFrame({    "age": [55, 60, 58, 65, 70],    "chol": [220, 240, 210, 280, 310],    "thalach": [150, 145, 158, 130, 120],})data["age_chol_product"] = data["age"] * data["chol"]print("Age x Cholesterol:\n", data[["age", "chol", "age_chol_product"]])data["max_hr_ratio"] = data["thalach"] / (220 - data["age"])print("\nMax heart rate ratio (thalach / predicted max):\n", data[["age", "thalach", "max_hr_ratio"]])

In [ ]:
# === Commented Template: Interaction Features ===# Uncomment and adapt to your own dataset.# import pandas as pd# data = pd.DataFrame({#     "feature_a": [val1, val2, val3],#     "feature_b": [val1, val2, val3],# })# data["a_times_b"] = data["feature_a"] * data["feature_b"]# EPS = 1e-6# data["a_over_b"] = data["feature_a"] / (data["feature_b"] + EPS)

In [ ]:
# TODO: Create new features based on clinical domain knowledge# The heart disease dataset combines patient demographics, blood work,# and ECG measurements. Interaction features can capture combined risk.## Examples:#   df["age_chol_interaction"] = df["age"] * df["chol"]#   df["max_hr_ratio"] = df["thalach"] / (220 - df["age"])## TODO: Create ratio features for ST depression# The slope of the ST segment and the magnitude of depression (oldpeak)# together indicate ischemia severity.#   df["oldpeak_slope"] = df["oldpeak"] * df["slope"]## TODO: Verify the new features# Check that they have finite values and reasonable ranges with .describe().

In [ ]:
# TODO: Save the engineered data for the next notebook# Include both the original and new features.PROCESSED_DIR = Path("../data/processed")PROCESSED_DIR.mkdir(parents=True, exist_ok=True)# df.to_csv(PROCESSED_DIR / "engineered_data.csv", index=False)# print("Engineered data saved to data/processed/engineered_data.csv")

### Unsupervised Clustering (KMeans)

Clustering groups observations without using the target labels.
We use **KMeans** which partitions data into $k$ clusters by minimizing
within-cluster variance.

**Questions:**
- Do the clusters found by KMeans align with the actual target classes?
- How many natural groups exist in the data?


In [ ]:
# TODO: Apply KMeans clustering and compare with target labels
# Clustering groups data without using labels.
# Use the elbow method to find optimal k, then compare clusters vs target.

# from sklearn.cluster import KMeans
# from sklearn.preprocessing import StandardScaler
# from sklearn.metrics import adjusted_rand_score
# from sklearn.decomposition import PCA

# Step 1: Scale features
# X_clust = df.drop("target", axis=1).select_dtypes(include=[np.number])
# X_clust_scaled = StandardScaler().fit_transform(X_clust)
#
# Step 2: Elbow method for k=2..10
# inertias = []
# for k in range(2, min(11, X_clust_scaled.shape[1] + 1)):
#     kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
#     kmeans.fit(X_clust_scaled)
#     inertias.append(kmeans.inertia_)
#
# Step 3: Plot elbow curve
# plt.plot(range(2, min(11, X_clust_scaled.shape[1] + 1)), inertias, 'bo-')
# plt.xlabel('k'); plt.ylabel('Inertia'); plt.title('Elbow Method')
# plt.show()
#
# Step 4: Fit KMeans and compare with target
# df["Cluster"] = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(X_clust_scaled)
# ari = adjusted_rand_score(df["target"], df["Cluster"])
# print(f"Adjusted Rand Index: {ari:.4f}")
# print(pd.crosstab(df["Cluster"], df["target"]))
#
# Step 5: Visualize via PCA
# pca_vis = PCA(n_components=2, random_state=42)
# X_pca_vis = pca_vis.fit_transform(X_clust_scaled)
# plt.scatter(X_pca_vis[:, 0], X_pca_vis[:, 1], c=df['Cluster'], cmap='viridis', edgecolors='k', alpha=0.7)
# plt.xlabel('PC1'); plt.ylabel('PC2'); plt.title('KMeans Clusters')
# plt.show()


### Principal Component Analysis (PCA)

PCA finds orthogonal axes (principal components) that capture the maximum variance
in the data. It is useful for:
- **Dimensionality reduction**: compressing many features into fewer components
- **Visualization**: projecting high-dimensional data to 2D or 3D
- **Noise reduction**: discarding low-variance components

PCA is **unsupervised** — it does not use the target labels.


In [ ]:
# TODO: Apply PCA for dimensionality reduction and visualization
# PCA finds the axes of maximum variance in the data.
# It can reduce high-dimensional data to 2D for visualization.

# from sklearn.decomposition import PCA
# from sklearn.preprocessing import StandardScaler

# X = df.drop("target", axis=1).select_dtypes(include=[np.number])
# if "Cluster" in X.columns:
#     X = X.drop("Cluster", axis=1)
# X_scaled = StandardScaler().fit_transform(X)
#
# Step 1: Fit PCA with min(n_features, 10) components
# n_comps = min(X.shape[1], 10)
# pca = PCA(n_components=n_comps, random_state=42)
# X_pca = pca.fit_transform(X_scaled)
#
# Step 2: Scree plot
# plt.bar(range(1, n_comps + 1), pca.explained_variance_ratio_)
# plt.xlabel('PC'); plt.ylabel('Explained Variance Ratio')
# plt.title('Scree Plot'); plt.show()
#
# Step 3: Cumulative explained variance
# cumulative = np.cumsum(pca.explained_variance_ratio_)
# plt.plot(range(1, n_comps + 1), cumulative, 'bo-')
# plt.axhline(y=0.95, color='r', linestyle='--', label='95%')
# plt.legend(); plt.show()
#
# Step 4: 2D PCA scatter colored by target
# plt.scatter(X_pca[:, 0], X_pca[:, 1], c=df["target"], cmap="coolwarm", edgecolors="k", alpha=0.7)
# plt.xlabel('PC1'); plt.ylabel('PC2')
# plt.title('PCA Projection'); plt.colorbar()
# plt.show()
#
# Step 5: Inspect feature loadings
# loadings = pd.DataFrame(
#     pca.components_.T[:, :3],
#     index=X.columns,
#     columns=['PC1', 'PC2', 'PC3'],
# )
# print(loadings['PC1'].abs().sort_values(ascending=False).head(5))


### Linear Discriminant Analysis (LDA)

LDA finds axes that **maximize class separability**. Unlike PCA (unsupervised),
LDA uses the target labels to find projections that best separate the classes.

**LDA vs PCA**:
- PCA maximizes **variance** (ignores labels)
- LDA maximizes **class separation** (uses labels)
- For classification, LDA often gives better separation in fewer components


In [ ]:
# TODO: Apply LDA for supervised dimensionality reduction
# LDA uses the target labels to maximize class separation.
# Compare its projection with PCA (unsupervised).

# from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

# X = df.drop("target", axis=1).select_dtypes(include=[np.number])
# if "Cluster" in X.columns:
#     X = X.drop("Cluster", axis=1)
# X_scaled = StandardScaler().fit_transform(X)
# y = df["target"]
#
# Step 1: Fit LDA
# n_classes = y.nunique()
# lda = LDA(n_components=min(n_classes - 1, X_scaled.shape[1]))
# X_lda = lda.fit_transform(X_scaled, y)
# print(f"LDA reduced to {X_lda.shape[1]} component(s)")
#
# Step 2: Visualize
# if X_lda.shape[1] >= 2:
#     plt.scatter(X_lda[:, 0], X_lda[:, 1], c=y, cmap='coolwarm', edgecolors='k', alpha=0.7)
#     plt.xlabel('LD1'); plt.ylabel('LD2')
# else:
#     for cls in sorted(y.unique()):
#         plt.hist(X_lda[y == cls, 0], bins=20, alpha=0.5, label=f'Class {cls}')
#     plt.legend(); plt.xlabel('LD1'); plt.ylabel('Frequency')
# plt.title('LDA Projection'); plt.show()
#
# Step 3: Side-by-side PCA vs LDA
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
# ax1.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='coolwarm', edgecolors='k', alpha=0.7)
# ax1.set_title('PCA (unsupervised)'); ax1.set_xlabel('PC1'); ax1.set_ylabel('PC2')
# x_lda_2d = X_lda[:, :min(2, X_lda.shape[1])]
# if x_lda_2d.shape[1] == 1:
#     x_lda_2d = np.hstack([x_lda_2d, np.zeros_like(x_lda_2d)])
# ax2.scatter(x_lda_2d[:, 0], x_lda_2d[:, 1], c=y, cmap='coolwarm', edgecolors='k', alpha=0.7)
# ax2.set_title('LDA (supervised)'); ax2.set_xlabel('LD1'); ax2.set_ylabel('LD2')
# plt.tight_layout(); plt.show()


### Recursive Feature Elimination (RFE)

RFE recursively removes the least important features, building a model at each step.
It ranks features by importance and finds the optimal subset.

**Benefits:**
- Reduces overfitting by removing noisy features
- Improves model interpretability
- Can speed up training and prediction


In [ ]:
# TODO: Apply RFE for feature selection
# RFE recursively removes the least important features.
# RFECV uses cross-validation to find the optimal subset.

# from sklearn.feature_selection import RFE, RFECV
# from sklearn.linear_model import LogisticRegression

# X = df.drop("target", axis=1).select_dtypes(include=[np.number])
# if "Cluster" in X.columns:
#     X = X.drop("Cluster", axis=1)
# X_scaled = StandardScaler().fit_transform(X)
# y = df["target"]
#
# Step 1: Fit RFE
# estimator = LogisticRegression(max_iter=1000, random_state=42)
# rfe = RFE(estimator=estimator, n_features_to_select=min(10, X_scaled.shape[1]), step=1)
# rfe.fit(X_scaled, y)
#
# Step 2: Display feature rankings
# ranking_df = pd.DataFrame({
#     'Feature': X.columns,
#     'Rank': rfe.ranking_,
#     'Selected': rfe.support_,
# }).sort_values('Rank')
# print(ranking_df)
#
# Step 3: RFECV for automatic feature count
# rfecv = RFECV(estimator=estimator, step=1, cv=5, scoring='accuracy')
# rfecv.fit(X_scaled, y)
# print(f"Optimal features: {rfecv.n_features_}")
#
# Step 4: Plot CV accuracy vs feature count
# plt.plot(range(len(rfecv.cv_results_['mean_test_score'])),
#          rfecv.cv_results_['mean_test_score'], 'bo-')
# plt.axvline(x=rfecv.n_features_, color='r', linestyle='--')
# plt.title('RFE: Optimal Feature Count'); plt.show()


### Exercises1. **Try different scalers**: Replace `StandardScaler` with `MinMaxScaler` or `RobustScaler` and compare how the scaled distributions look.2. **Identify redundant features**: Find pairs of features with correlation > 0.9 (e.g., oldpeak and slope). Would dropping one affect model performance?3. **Create a cholesterol-to-age ratio**: Compute `df["chol_per_year"] = df["chol"] / df["age"]` — do older patients have higher cholesterol per year of age?4. **Log transform skewed features**: Features like `chol` or `trestbps` may be right-skewed. Try `np.log1p()` and check if the distribution becomes more normal.5. **Pairplot**: Use `sns.pairplot()` on a subset of the most discriminative features, coloring by target — do healthy and heart disease patients form distinct clusters?